In [23]:
!python v20240512_generate_h5_data_with_scales_and_chains.py

No. of Matched Files: 256
No. of Missing Pairs: 0
0it [00:00, ?it/s]a 2Y9J_backbone
21420.0
1it [00:01,  1.33s/it]J 2YEW_backbone
10488.0
2it [00:01,  1.12it/s]C 3IXV_backbone
33696.0
3it [00:03,  1.05s/it]B 3IYJ_backbone
79002.0
4it [00:03,  1.26it/s]F 3IZI_backbone
42624.0
5it [00:04,  1.08it/s]B 3J17_backbone
91728.0
6it [00:05,  1.24it/s]B 3J1P_backbone
101871.0
7it [00:05,  1.63it/s]A 3J22_backbone
129360.0
8it [00:05,  2.19it/s]O 3J2W_backbone
26730.0
9it [00:06,  2.07it/s]B 3J7V_backbone
46342.0
10it [00:06,  2.35it/s]C 3J94_backbone
89792.0
11it [00:06,  2.42it/s]D 3J9P_backbone
126000.0
12it [00:07,  2.66it/s]C 3J9T_backbone
26136.0
13it [00:08,  1.72it/s]c 3JC5_backbone
41514.0
14it [00:08,  1.59it/s]A 3JCL_backbone
155925.0
15it [00:09,  1.72it/s]O 3JD6_backbone
250047.0
B 4BTG_backbone
81972.0
17it [00:09,  2.71it/s]C 4CG5_backbone
27744.0
A 4PT2_backbone
87438.0
19it [00:09,  3.77it/s]D 4UQQ_backbone
130032.0
20it [00:10,  3.45it/s]E 4V1W_backbone
26598.0
21it [00:10,  2.7

In [2]:
!ls -1 ../data/full_pdb_homologs_new/| wc -l

256


In [1]:
!ls -1 ../data/full_pdb_homologs/| wc -l

256


In [9]:
%load_ext autoreload
%autoreload 2

In [5]:
from v20240507_generate_h5_data_with_scales import *

In [11]:
# Example usage
backbone_dir = '../data/backbones'
homolog_dir = '../data/full_pdb_homologs'
output_file = './data/20240507_cryo_data_with_scales.h5'

matched_files, missing_pairs = match_files(backbone_dir, homolog_dir)
print("No. of Matched Files:", len(matched_files))
print("No. of Missing Pairs:", len(missing_pairs))

for backbone_file, homolog_file in tqdm(zip(sorted(os.listdir(backbone_dir)), sorted(os.listdir(homolog_dir)))):
    true_ca_coords = parse_ca_atoms(os.path.join(backbone_dir, backbone_file))
    homolog_ca_coords = parse_ca_atoms(os.path.join(homolog_dir, homolog_file))
    
    # Convert coordinates to binary grids
    # Changing to 64^3 here
    true_scale, true_ca = coords_to_binary_grid(true_ca_coords, (64,64,64))
    homolog_scale, homolog_ca = coords_to_binary_grid(homolog_ca_coords)
    _, true_vol = create_gaussian_volume(true_ca)  
    break

No. of Matched Files: 256
No. of Missing Pairs: 0


0it [00:01, ?it/s]

(8544, 3)
(3,)
(3,)
(8544, 3)
(3,)
(3,)


array([63, 63, 63])

In [38]:
import torch
import numpy as np
from torch.utils.data import Dataset
import h5py

class CryoData(Dataset):
    def __init__(self, h5_file):
        self.h5_file = h5_file
        with h5py.File(self.h5_file, 'r') as file:
            self.keys = list(file.keys())

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as file:
            key_name = self.keys[idx]
            group = file[key_name]
            true_ca = torch.tensor(group['true_ca'][:])
            homolog_ca = torch.tensor(group['homolog_ca'][:])
            true_vol = torch.tensor(group['true_vol'][:])
            true_scale = torch.tensor(group['true_scale'][:])
            homolog_scale = torch.tensor(group['homolog_scale'][:])
            # scale_factors = torch.tensor(group['scale_factors'][:])
        return {'name': key_name[:4],
                'true_ca': true_ca, 
                'homolog_ca': homolog_ca, 
                'true_vol': true_vol,
                'true_scale': true_scale, 
                'homolog_scale': homolog_scale}

# Usage example
dataset = CryoData('./data/20240507_cryo_data_with_scales.h5')

In [39]:

# Assuming you have a CryoData instance called 'dataset'
for i in range(1):  # Check the first three samples
    sample = dataset[i]
    print(f"Sample {i}:")
    print(f"looking at {sample['name']}")
    print(sample)

Sample 0:
looking at 2Y9J
{'name': '2Y9J', 'true_ca': tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         